# DATASCI 503, Group Work 7: Trees and Tree Ensembles

**Instructions:** During lab section, and afterward as necessary, you will collaborate in two-person teams (assigned by the GSI) to complete the problems that are interspersed below. The GSI will help individual teams encountering difficulty, make announcements addressing common issues, and help ensure progress for all teams. During lab, feel free to flag down your GSI to ask questions at any point!

## Classification Trees

A **classification tree** is built through a process known as binary recursive partitioning. The main idea is as follows:
* Recursively partition the input space into rectangular boxes
* At each step, ask a question about one variable (split the feature space into parts)
* Repeat for each branch (recursively partition the feature space into boxes)
* Goal: each box should contain data points mostly from the same class
* Each box is labelled with its majority class
* The end result: a tree of splits, a partitioning of the variable space into boxes and assignment of a class label to each box



Scikit-learn implements CART, whose fundamental principles and methodologies behind its decision tree algorithms are largely based on the concepts introduced by Breiman et al. in the book Classification and Regression Trees.

Additionally, scikit-learn enhances the basic CART algorithm with several modern features like support for missing values, various criteria for splitting (Gini impurity, entropy for classification, and mean squared error, mean absolute error for regression), and pre-pruning options.

In [ ]:
import io
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from sklearn import tree
from sklearn.ensemble import (
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    mean_squared_error,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

import os
import os
if os.path.isdir("fixtures"):
    !cp -n fixtures/* .

---

**Problem 1: Variable Setup and Selection**

We will work with the NHANES dataset throughout this lab. Please do the following:

1. Load the NHANES datasets and merge them on `SEQN`.
2. Use the following features: Gender, Age, Weight, Height, BMI, WaistSize, HouseholdSize, and Ethnicity, along with the HDL cholesterol level. Refer to the docs for variable names:
   - [HDL_L](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/HDL_L.htm)
   - [DEMO_L](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DEMO_L.htm)
   - [BMX_L](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BMX_L.htm)
3. Rename variables to English-readable Python style (e.g., `RIAGENDR` becomes `Gender`).
4. Drop all rows with missing values.

Store the result in `my_df`.

In [ ]:
# BEGIN SOLUTION

In [ ]:
# Test assertions
assert "my_df" in dir(), "my_df should be defined"
assert len(my_df.columns) == 9, f"Expected 9 columns, got {len(my_df.columns)}"
assert "HDL" in my_df.columns, "HDL column should be present"
assert my_df.isna().sum().sum() == 0, "There should be no missing values"
print("All tests passed!")

---

**Problem 2: Train/Test Split**

Split your data into train and test sets with an 80%/20% breakdown.

**Use `random_state=42` for this and all subsequent problems.**

Store your splits in `X_train`, `X_test`, `y_train`, and `y_test`.

In [ ]:
# BEGIN SOLUTION

In [ ]:
# Test assertions
total_samples = len(my_df)
train_ratio = len(X_train) / total_samples
test_ratio = len(X_test) / total_samples
assert 0.78 < train_ratio < 0.82, f"Train ratio should be ~80%, got {train_ratio:.2%}"
assert 0.18 < test_ratio < 0.22, f"Test ratio should be ~20%, got {test_ratio:.2%}"
print("All tests passed!")

---

## Classification Trees on the Spam Dataset

We look at the spam email classification problem. This dataset is available at the [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/spambase).

In [ ]:
url = "https://archive.ics.uci.edu/static/public/94/spambase.zip"
response = requests.get(url, timeout=30)
z = zipfile.ZipFile(io.BytesIO(response.content))
z.extractall(".")

In [ ]:
column_names = [
    "word_freq_make", "word_freq_address", "word_freq_all", "word_freq_3d",
    "word_freq_our", "word_freq_over", "word_freq_remove", "word_freq_internet",
    "word_freq_order", "word_freq_mail", "word_freq_receive", "word_freq_will",
    "word_freq_people", "word_freq_report", "word_freq_addresses", "word_freq_free",
    "word_freq_business", "word_freq_email", "word_freq_you", "word_freq_credit",
    "word_freq_your", "word_freq_font", "word_freq_000", "word_freq_money",
    "word_freq_hp", "word_freq_hpl", "word_freq_george", "word_freq_650",
    "word_freq_lab", "word_freq_labs", "word_freq_telnet", "word_freq_857",
    "word_freq_data", "word_freq_415", "word_freq_85", "word_freq_technology",
    "word_freq_1999", "word_freq_parts", "word_freq_pm", "word_freq_direct",
    "word_freq_cs", "word_freq_meeting", "word_freq_original", "word_freq_project",
    "word_freq_re", "word_freq_edu", "word_freq_table", "word_freq_conference",
    "char_freq_;", "char_freq_(", "char_freq_[", "char_freq_!",
    "char_freq_$", "char_freq_#",
    "capital_run_length_average", "capital_run_length_longest",
    "capital_run_length_total", "is_spam",
]

In [ ]:
spam_data = pd.read_csv("spambase.data", names=column_names)

In the dataset, there are 57 continuous input variables and 1 binary outcome variable:

- `word_freq_WORD`: percentage of words matching WORD (48 variables, values in [0,100])
- `char_freq_CHAR`: percentage of characters matching CHAR (6 variables, values in [0,100])
- `capital_run_length_average/longest/total`: statistics on capital letter runs
- `is_spam`: 1 = spam, 0 = not spam

In [ ]:
spam_train, spam_test = train_test_split(
    spam_data, test_size=0.3, random_state=1, stratify=spam_data["is_spam"]
)

Let us take a look at the word frequencies. Words appearing too frequently or too rarely can be useless for making predictions.

In [ ]:
word_freq_mean = spam_train.mean(axis=0)[0:48]
word_freq_mean_sort = word_freq_mean.sort_values(ascending=True)
word_freq_mean_sort

It's interesting to see that the word `george` has high frequency. This is because George is the donor of the dataset. We may see later that `hp`, the place George works, is selected by tree model. Using those words may cause generalization problems. We also find the labels are slightly unbalanced.

In [ ]:
pd.crosstab(index=spam_train["is_spam"], columns="count")

You can check the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) for `DecisionTreeClassifier`. Key arguments:

- `criterion`: {"gini", "entropy", "log_loss"}, default="gini"
- `class_weight`: dict, list of dict or "balanced", default=None
- `ccp_alpha`: non-negative float, default=0.0 (complexity parameter for pruning)

In [ ]:
X_spam_train = spam_train.drop(["is_spam"], axis=1)
y_spam_train = spam_train["is_spam"]
X_spam_test = spam_test.drop(["is_spam"], axis=1)
y_spam_test = spam_test["is_spam"]

In [ ]:
tree1 = DecisionTreeClassifier(random_state=0)
tree1 = tree1.fit(X_spam_train, y_spam_train)

In [ ]:
training_error = 1 - tree1.score(X_spam_train, y_spam_train)
test_error = 1 - tree1.score(X_spam_test, y_spam_test)
print("training error is", training_error)
print("test error is", test_error)

One of the most attractive properties of trees is that they can be graphically displayed. We use the `plot_tree()` function to depict the tree structure.

In [ ]:
plt.figure(figsize=(20, 10))
tree.plot_tree(tree1)
plt.show()

Since we didn't penalize the size of the tree, we generated an extremely complex model. As mentioned in lecture, we prune a tree by finding a sub-tree $T$ that minimizes:
$$
  C(T)=\sum_{t=1}^{|T|} N_{t} \cdot \text{Impurity} (t)+c_{p} \cdot|T|.
$$

## Comparing Different Splitting Criteria

Previously we used Gini index. Now we try entropy ($-\sum_{k=1}^{K} p_{k}(m) \log p_{k}(m)$).

In [ ]:
tree2 = DecisionTreeClassifier(random_state=0, criterion="entropy")
tree2 = tree2.fit(X_spam_train, y_spam_train)

training_error = 1 - tree2.score(X_spam_train, y_spam_train)
test_error = 1 - tree2.score(X_spam_test, y_spam_test)
print("training error is", training_error)
print("test error is", test_error)

Using cross-entropy, we get smaller test error compared with Gini index.

## Class Weighting

There are two types of misclassification: predicting Spam as Non-Spam and vice versa. Our model tends to misclassify Spam as Non-Spam since Non-Spam is the dominant class. We can assign weights to adjust for this imbalance.

In [ ]:
y_spam_train.value_counts()

In [ ]:
proportion_non_spam = y_spam_train.value_counts()[0] / y_spam_train.shape[0]
proportion_spam = y_spam_train.value_counts()[1] / y_spam_train.shape[0]
proportion_non_spam / proportion_spam

In [ ]:
# Model without balancing labels
tree2_no_weight = DecisionTreeClassifier(random_state=0)
tree2_no_weight.fit(X_spam_train, y_spam_train)

training_error = 1 - tree2_no_weight.score(X_spam_train, y_spam_train)
test_error = 1 - tree2_no_weight.score(X_spam_test, y_spam_test)
print("training error is", training_error)
print("test error is", test_error)

y_pred_test = tree2_no_weight.predict(X_spam_test)
cm = confusion_matrix(y_spam_test, y_pred_test)
cm_display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=tree2_no_weight.classes_)
cm_display.plot();

In [ ]:
# Model with balancing labels
tree2_weight = DecisionTreeClassifier(
    random_state=0, class_weight={0: 1, 1: proportion_non_spam / proportion_spam}
)
tree2_weight.fit(X_spam_train, y_spam_train)

training_error = 1 - tree2_weight.score(X_spam_train, y_spam_train)
test_error = 1 - tree2_weight.score(X_spam_test, y_spam_test)
print("training error is", training_error)
print("test error is", test_error)

y_pred_test = tree2_weight.predict(X_spam_test)
cm = confusion_matrix(y_spam_test, y_pred_test)
cm_display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=tree2_weight.classes_)
cm_display.plot();

---

**Problem 3: Motivation**

Which of the following best describes when decision tree methods are most likely to outperform linear regression?

- **A.** When there are many features but few observations
- **B.** When the relationship between features and target is approximately linear
- **C.** When the underlying relationships involve nonlinear patterns or complex interactions between features
- **D.** When the dataset has no missing values

Set your answer in the variable `answer_3` (e.g., `answer_3 = "A"`).

In [ ]:
# BEGIN SOLUTION
answer_3 = ""

In [ ]:
# Test assertions
assert "answer_3" in dir(), "answer_3 should be defined"
assert answer_3 in ("A", "B", "C", "D"), "answer_3 should be one of A, B, C, D"
print("All tests passed!")

## Regression Trees

In [ ]:
# Create a random dataset
rng = np.random.RandomState(1)
X_demo = np.sort(200 * rng.rand(100, 1) - 100, axis=0)
y_demo = np.array([np.pi * np.sin(X_demo).ravel(), np.pi * np.cos(X_demo).ravel()]).T
y_demo[::5, :] += 0.5 - rng.rand(20, 2)

regr_1 = DecisionTreeRegressor(max_depth=2)
regr_2 = DecisionTreeRegressor(max_depth=5)
regr_3 = DecisionTreeRegressor(max_depth=8)
regr_1.fit(X_demo, y_demo)
regr_2.fit(X_demo, y_demo)
regr_3.fit(X_demo, y_demo)

X_demo_test = np.arange(-100.0, 100.0, 0.01)[:, np.newaxis]
y_1 = regr_1.predict(X_demo_test)
y_2 = regr_2.predict(X_demo_test)
y_3 = regr_3.predict(X_demo_test)

plt.figure()
s = 25
plt.scatter(y_demo[:, 0], y_demo[:, 1], c="navy", s=s, edgecolor="black", label="data")
plt.scatter(y_1[:, 0], y_1[:, 1], c="cornflowerblue", s=s, edgecolor="black", label="max_depth=2")
plt.scatter(y_2[:, 0], y_2[:, 1], c="red", s=s, edgecolor="black", label="max_depth=5")
plt.scatter(y_3[:, 0], y_3[:, 1], c="orange", s=s, edgecolor="black", label="max_depth=8")
plt.xlim([-6, 6])
plt.ylim([-6, 6])
plt.xlabel("target 1")
plt.ylabel("target 2")
plt.title("Multi-output Decision Tree Regression")
plt.legend(loc="best")
plt.show()

---

**Problem 4: CART Regression on NHANES**

Train a `DecisionTreeRegressor` to predict HDL levels on the NHANES data. Evaluate the model on train and test sets using mean squared error (MSE).

Store results in `train_mse_cart` and `test_mse_cart`. Use `random_state=42`.

In [ ]:
# BEGIN SOLUTION

print(f"CART - Train: {train_mse_cart:.2f}, Test: {test_mse_cart:.2f}")

In [ ]:
# Test assertions
assert train_mse_cart >= 0
assert test_mse_cart > train_mse_cart
print("All tests passed!")


## Random Forests

A **random forest** is an ensemble method that combines multiple CART models. Each model is fitted using a bootstrapped sample, and at each split node, only a subset of $m$ predictors are chosen as candidates, typically $m \approx \sqrt{p}$ for classification and $m \approx p/3$ for regression. This *decorrelates* the trees.

We can adjust the number of predictors via `max_features`.

Setting `max_features` equal to the total number of features gives us bagging (no feature restriction).

In [ ]:
# Bagging (all features)
clf = RandomForestClassifier(random_state=0, max_features=X_spam_train.shape[1])
clf.fit(X_spam_train, y_spam_train)
test_error_bagging = 1 - clf.score(X_spam_test, y_spam_test)
print(f"Bagging test error: {test_error_bagging:.4f}")

In [ ]:
# Random Forest (sqrt features, default)
rf = RandomForestClassifier(random_state=0)
rf.fit(X_spam_train, y_spam_train)
test_error_rf = 1 - rf.score(X_spam_test, y_spam_test)
print(f"Random Forest test error: {test_error_rf:.4f}")

### Feature Importance (Mean Decrease in Impurity)

Feature importances are computed as the mean accumulation of the impurity decrease within each tree. Note that impurity-based importances can be misleading for high cardinality features.

In [ ]:
importances = rf.feature_importances_
forest_importances = pd.Series(importances, index=rf.feature_names_in_)

plt.figure(figsize=(10, 12))
forest_importances.sort_values(ascending=True).plot.barh()
plt.ylabel("Mean decrease in impurity")
plt.show()

### Feature Importance (Permutation-Based)

Permutation feature importance overcomes limitations of impurity-based importance: no bias toward high-cardinality features, and can be computed on a held-out test set.

In [ ]:
result = permutation_importance(
    rf, X_spam_test, y_spam_test, n_repeats=10, random_state=42, n_jobs=2
)

forest_importances = pd.Series(result.importances_mean, index=rf.feature_names_in_)
plt.figure(figsize=(10, 12))
forest_importances.sort_values(ascending=True).plot.barh()
plt.ylabel("Mean decrease in accuracy")
plt.show()

We observe some negative values for permutation importances. In those cases, the predictions on the shuffled data happened to be more accurate than the real data. This happens when the feature didn't matter.

---

**Problem 5: Random Forest Regression on NHANES**

Train a `RandomForestRegressor` to predict HDL levels on the NHANES data. Evaluate on train and test sets using MSE.

Store results in `train_mse_rf` and `test_mse_rf`. Use `random_state=42`.

In [ ]:
# BEGIN SOLUTION

print(f"RF - Train: {train_mse_rf:.2f}, Test: {test_mse_rf:.2f}")

In [ ]:
# Test assertions
assert train_mse_rf >= 0
assert test_mse_rf >= 0
print("All tests passed!")


## Boosting

By using CART as a base learner, gradient boosting trains new iterations based on the errors of previous models:

$$f_m(x)=f_{m-1}(x)+\left(\underset{h_m \epsilon H}{\operatorname{argmin}}\left[\sum_{i=1}^N L\left(y_i, f_{m-1}\left(x_i\right)+h_m\left(x_i\right)\right)\right]\right)(x)$$

Each $h_m$ is a weak learner trained to minimize the remaining error after $f_{m-1}$ is learned.

In [ ]:
clf = GradientBoostingClassifier(
    n_estimators=1000, random_state=0, max_features=X_spam_train.shape[1]
)
clf.fit(X_spam_train, y_spam_train)
test_error_boosting = 1 - clf.score(X_spam_test, y_spam_test)
print(f"Boosting (all features) test error: {test_error_boosting:.4f}")

In [ ]:
clf = GradientBoostingClassifier(n_estimators=1000, random_state=0, max_features="sqrt")
clf.fit(X_spam_train, y_spam_train)
test_error_boosting = 1 - clf.score(X_spam_test, y_spam_test)
print(f"Boosting (sqrt features) test error: {test_error_boosting:.4f}")

---

**Problem 6: Boosting Regression + Comparison**

Train a `GradientBoostingRegressor` on the NHANES data. Evaluate on train and test sets using MSE. Use `random_state=42`.

Store results in `train_mse_boosting` and `test_mse_boosting`.

Then create a DataFrame called `results_df` comparing all three models and use the following names for indexes and columns:
- **Index**: Train Error, Test Error
- **Columns**: CART, Random Forest, Boosting
- **Values**: The corresponding MSE values

In [ ]:
# BEGIN SOLUTION


print(results_df)

In [ ]:
# Test assertions
assert train_mse_boosting >= 0
assert isinstance(results_df, pd.DataFrame)
assert results_df.shape == (2, 3), f"Expected (2,3), got {results_df.shape}"
assert list(results_df.columns) == ["CART", "Random Forest", "Boosting"]
print("All tests passed!")


---

## Classification on NHANES

**Problem 7: Switching to Classification**

HDL levels can be categorized ([Cleveland Clinic](https://my.clevelandclinic.org/health/articles/11920-cholesterol-numbers-what-do-they-mean)):

- **Heart-Healthy (0)**: HDL >= 60 mg/dL
- **At-Risk (1)**: HDL 40-59 for men, or HDL 50-59 for women
- **Dangerous (2)**: HDL < 40 for men, or HDL < 50 for women

Create a `Level` column in `my_df` with these labels (0, 1, or 2). Then create new train/test splits (80/20) **stratified on `Level`**. Use `random_state=42`.

**Note**: Gender is encoded as 1 = male, 2 = female in NHANES.

In [ ]:
# BEGIN SOLUTION


In [ ]:
# Test assertions
assert "Level" in my_df.columns
assert set(my_df['Level'].unique()).issubset({0, 1, 2})

heart_healthy = my_df[my_df["HDL"] >= 60]
assert (heart_healthy["Level"] == 0).all()
dangerous_males = my_df[(my_df["Gender"] == 1) & (my_df["HDL"] < 40)]
assert (dangerous_males["Level"] == 2).all()
dangerous_females = my_df[(my_df["Gender"] == 2) & (my_df["HDL"] < 50)]
assert (dangerous_females["Level"] == 2).all()
print("All tests passed!")

---

**Problem 8: Classification Models**

Train CART (`DecisionTreeClassifier`), Random Forest (`RandomForestClassifier`), and Gradient Boosting (`GradientBoostingClassifier`) for the classification task. Evaluate using accuracy.

Store results as `train_acc_cart`, `test_acc_cart`, `train_acc_rf`, `test_acc_rf`, `train_acc_boosting`, `test_acc_boosting`. Use `random_state=42`.

In [ ]:
# BEGIN SOLUTION


print(f"CART  - Train: {train_acc_cart:.3f}, Test: {test_acc_cart:.3f}")
print(f"RF    - Train: {train_acc_rf:.3f}, Test: {test_acc_rf:.3f}")
print(f"Boost - Train: {train_acc_boosting:.3f}, Test: {test_acc_boosting:.3f}")

In [ ]:
# Test assertions
assert 0 <= train_acc_cart <= 1
assert 0 <= test_acc_cart <= 1
assert 0 <= train_acc_rf <= 1
assert 0 <= test_acc_boosting <= 1
print("All tests passed!")

---

**Problem 9: Classification Results Display**

Create a DataFrame called `classification_results_df` comparing classification performance:
- **Index**: Train Accuracy, Test Accuracy
- **Columns**: CART, Random Forest, Boosting

In [ ]:
# BEGIN SOLUTION

print(classification_results_df)

In [ ]:
# Test assertions
assert isinstance(classification_results_df, pd.DataFrame)
assert classification_results_df.shape == (2, 3)
assert list(classification_results_df.columns) == ["CART", "Random Forest", "Boosting"]
print("All tests passed!")


---

**Problem 10: Error Costs**

In the HDL classification task with levels Heart-Healthy (0), At-Risk (1), and Dangerous (2), which misclassification is the most harmful from a health perspective?

- **A.** Predicting At-Risk (1) when the truth is Heart-Healthy (0)
- **B.** Predicting Dangerous (2) when the truth is At-Risk (1)
- **C.** Predicting Heart-Healthy (0) when the truth is Dangerous (2)
- **D.** Predicting At-Risk (1) when the truth is Dangerous (2)

Set your answer in `answer_10`.

In [ ]:
# BEGIN SOLUTION
answer_10 = ""

In [ ]:
# Test assertions
assert "answer_10" in dir()
assert answer_10 in ("A", "B", "C", "D")
print("All tests passed!")


---

**Problem 11: Error Analysis**

The worst misclassification is predicting Heart-Healthy (0) when the truth is Dangerous (2) — this corresponds to entry `[2, 0]` of the confusion matrix.

Compute the confusion matrix for each model and create a **bar chart** showing the count of this worst error for each model. Label the x-axis `"Model"`, the y-axis `"Count"`, and title it `"Dangerous to Heart-Healthy Misclassification by Model"`.

Store the figure in `fig_err`, the axes in `ax_err`, and the name of the model that commits this error the **least** in `best_model_for_safety` (one of `"CART"`, `"Random Forest"`, or `"Boosting"`).

In [ ]:
# BEGIN SOLUTION

In [ ]:
# Test assertions
assert "fig_err" in dir(), "fig_err should be defined"
assert "ax_err" in dir(), "ax_err should be defined"
assert "best_model_for_safety" in dir(), "best_model_for_safety should be defined"
assert len(ax_err.patches) == 3, "Should have 3 bars"
print("All tests passed!")

# BEGIN HIDDEN TESTS

---

**Problem 12: Maximizing Performance**

Your task is to achieve a test classification accuracy of **57% or higher**.

A validation set has been created from the training data below. Use it to tune your hyperparameters — try different settings and pick the configuration with the best validation accuracy. Only look at test accuracy once you have settled on your final model.

You may use any decision tree ensemble method. Hyperparameters you can tune include:
- Learning rate, number of estimators, tree depth, `min_samples_leaf`, `max_features`, `min_impurity_decrease`

Train your final model on the full `X_train`/`y_train` and store it in `final_model`.

**Important**: Do not include HDL as a feature.

**Hint**: See the [GradientBoostingClassifier documentation](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html).

In [ ]:
# Validation split for hyperparameter tuning — do NOT use X_val/y_val for final evaluation
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42, stratify=y_train
)

In [ ]:
# Tune your hyperparameters here using X_tr/y_tr for training and X_val/y_val for selection
# Example:
# model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
# model.fit(X_tr, y_tr)
# print(model.score(X_val, y_val))

# BEGIN SOLUTION

# END SOLUTION

final_accuracy = final_model.score(X_test, y_test)
print(f"Test Accuracy: {final_accuracy:.4f}")

In [ ]:
# Test assertions
assert "final_model" in dir(), "final_model should be defined"
final_accuracy = final_model.score(X_test, y_test)
assert final_accuracy > 0.57, f"Test accuracy should be > 57%, got {final_accuracy:.2%}"
print(f"Congratulations! Your model achieved {final_accuracy:.2%} accuracy.")
print("All tests passed!")